In [1]:
import json
import pickle
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [2]:
N = 2000
 
# Lookup Dimensions
SATISFACTION_LEVELS = {1:"Very Dissatisfied",2:"Dissatisfied",3:"Neutral",
                        4:"Satisfied",5:"Very Satisfied"}
RATING_LEVELS = {1:"Unacceptable",2:"Needs Improvement",3:"Meets Expectation",
                  4:"Exceeds Expectation",5:"Above and Beyond"}
EDUCATION_LEVELS = {1:"No Formal Qualifications",2:"High School",
                     3:"Bachelor",4:"Master",5:"Doctorate"}
 
df_satisfaction = pd.DataFrame({"SatisfactionID":list(SATISFACTION_LEVELS.keys()),
                                  "SatisfactionLevel":list(SATISFACTION_LEVELS.values())})
df_rating  = pd.DataFrame({"RatingID":list(RATING_LEVELS.keys()),
                             "RatingLevel":list(RATING_LEVELS.values())})
df_education = pd.DataFrame({"EducationLevelID":list(EDUCATION_LEVELS.keys()),
                               "EducationLevel":list(EDUCATION_LEVELS.values())})

In [3]:
# Org Pyramid
levels      = ["L1","L2","L3","L4","L5","L6"]
level_probs = [0.35,0.28,0.18,0.11,0.06,0.02]
level_salary = {"L1":400000,"L2":650000,"L3":950000,"L4":1400000,"L5":2000000,"L6":3000000}
dept_multiplier = {"Engineering":1.20,"Finance":1.10,"Legal":1.15,
                    "Sales":0.95,"Marketing":0.95,"HR":0.90,"Operations":0.85}
 
dept_map = {"Engineering":0.35,"Sales":0.20,"HR":0.08,"Finance":0.10,
             "Marketing":0.10,"Operations":0.12,"Legal":0.05}
departments, dept_weights = list(dept_map.keys()), list(dept_map.values())
 
locations  = ["Bengaluru","Mumbai","Hyderabad","Pune","Chennai","Delhi","Remote"]
loc_probs  = [0.30,0.20,0.15,0.12,0.10,0.08,0.05]
 
first_names = ["Aarav","Priya","Rohan","Sneha","Vikram","Ananya","Arjun","Kavya",
               "Dev","Meera","Siddharth","Pooja","Rahul","Nisha","Kiran","Divya"]
last_names  = ["Sharma","Patel","Kumar","Singh","Mehta","Gupta","Joshi","Nair",
               "Rao","Iyer","Reddy","Verma","Das","Bose","Pillai"]

In [5]:
from datetime import datetime, timedelta

hire_start = datetime(2010,1,1); hire_end = datetime(2023,12,31)
ref_date   = datetime(2024,6,30)
hire_dates = [hire_start + timedelta(days=int(x))
              for x in np.random.uniform(0,(hire_end-hire_start).days,N)]
years_at_company = [(ref_date-h).days/365 for h in hire_dates]

In [6]:
# Generate columns
level       = np.random.choice(levels,N,p=level_probs)
location    = np.random.choice(locations,N,p=loc_probs)
department  = np.random.choice(departments,N,p=dept_weights)
gender      = np.random.choice(["Male","Female","Non-Binary"],N,p=[0.52,0.44,0.04])
age         = np.clip(np.random.normal(35,9,N).astype(int),22,60)
education_id= np.random.choice([1,2,3,4,5],N,p=[0.03,0.10,0.45,0.32,0.10])
marital     = np.random.choice(["Single","Married","Divorced"],N,p=[0.35,0.50,0.15])
ethnicity   = np.random.choice(["Asian","White","Hispanic","Black","Other"],N,
                                p=[0.55,0.20,0.10,0.08,0.07])
state       = np.random.choice(["Karnataka","Maharashtra","Telangana","Tamil Nadu","Delhi","Goa"],N)
biz_travel  = np.random.choice(["None","Rarely","Frequently"],N,p=[0.30,0.45,0.25])
distance_km = np.clip(np.random.exponential(15,N).astype(int),1,100)
overtime    = np.random.choice(["Yes","No"],N,p=[0.28,0.72])
stock_opt   = np.random.choice([0,1,2,3],N,p=[0.35,0.35,0.20,0.10])
salary      = np.array([int(level_salary[l]*dept_multiplier[d]*np.random.uniform(0.85,1.15))
                         for l,d in zip(level,department)])
market_rate = np.clip(np.random.normal(1.0,0.15,N),0.60,1.50)
 
years_in_role   = np.array([min(y*np.random.uniform(0.2,1.0),y) for y in years_at_company])
yrs_since_promo = np.array([min(y*np.random.uniform(0.0,0.8),y) for y in years_at_company])
yrs_curr_mgr    = np.array([min(y*np.random.uniform(0.1,0.9),y) for y in years_at_company])

In [7]:
# Leave windows
leave_30d  = np.random.poisson(0.8,N)
leave_90d  = leave_30d  + np.random.poisson(1.5,N)
leave_180d = leave_90d  + np.random.poisson(2.0,N)
leave_365d = leave_180d + np.random.poisson(3.0,N)
awards     = np.random.choice([0,1,2,3,4],N,p=[0.45,0.30,0.15,0.07,0.03])
 

In [8]:
# Performance fact
env_sat = np.random.randint(1,6,N)
job_sat = np.random.randint(1,6,N)
rel_sat = np.random.randint(1,6,N)
wlb     = np.random.randint(1,6,N)
s_rate  = np.random.randint(1,6,N)
m_rate  = np.clip(s_rate+np.random.randint(-1,2,N),1,5)
tr_avail = np.random.randint(2,8,N)
tr_taken = np.array([np.random.randint(0,t+1) for t in tr_avail])

In [9]:
# Attrition label – logistic function of domain-weighted risk factors
risk = (
    -1.5*market_rate +0.8*(overtime=="Yes").astype(int)
    -0.5*(job_sat/5) -0.4*(env_sat/5) -0.3*(wlb/5)
    +0.6*(yrs_since_promo/(np.array(years_at_company)+0.1))
    +0.4*(leave_90d/5) -0.3*(awards/4)
    +0.3*(biz_travel=="Frequently").astype(int)
    +0.2*(distance_km/100) + np.random.normal(0,0.5,N)
)
attrition_prob_true = 1/(1+np.exp(-risk))
attrition = (np.random.uniform(0,1,N)<attrition_prob_true).astype(int)
print(f"Dataset: {N} employees | Attrition rate: {attrition.mean():.1%}")
 

Dataset: 2000 employees | Attrition rate: 19.2%


In [10]:
# Assemble DataFrames
emp_ids = [f"EMP{str(i+1).zfill(5)}" for i in range(N)]
 
df_emp = pd.DataFrame({
    "EmployeeID":emp_ids, "FirstName":np.random.choice(first_names,N),
    "LastName":np.random.choice(last_names,N), "Gender":gender, "Age":age,
    "Level":level, "Location":location, "Department":department,
    "BusinessTravel":biz_travel, "DistanceFromHome_KM":distance_km,
    "State":state, "Ethnicity":ethnicity, "MaritalStatus":marital,
    "Salary":salary, "MarketRateRatio":market_rate.round(3),
    "StockOptionLevel":stock_opt, "OverTime":overtime,
    "HireDate":[h.strftime("%Y-%m-%d") for h in hire_dates],
    "Attrition":["Yes" if a else "No" for a in attrition],
    "YearsAtCompany":np.round(years_at_company,2),
    "YearsInMostRecentRole":np.round(years_in_role,2),
    "YearsSinceLastPromotion":np.round(yrs_since_promo,2),
    "YearsWithCurrManager":np.round(yrs_curr_mgr,2),
    "EducationLevelID":education_id,
    "Leave_Last30Days":leave_30d, "Leave_Last90Days":leave_90d,
    "Leave_Last180Days":leave_180d, "Leave_Last365Days":leave_365d,
    "AwardsReceived":awards,
})
df_perf = pd.DataFrame({
    "PerformanceID":[f"PR{str(i+1).zfill(5)}" for i in range(N)],
    "EmployeeID":emp_ids,
    "ReviewDate":[(ref_date-timedelta(days=int(np.random.uniform(0,180)))).strftime("%Y-%m-%d") for _ in range(N)],
    "EnvironmentSatisfaction":env_sat, "JobSatisfaction":job_sat,
    "RelationshipSatisfaction":rel_sat, "WorkLifeBalance":wlb,
    "SelfRating":s_rate, "ManagerRating":m_rate,
    "TrainingOpportunitiesWithinYear":tr_avail, "TrainingOpportunitiesTaken":tr_taken,
})
 
# Merge all
df = df_emp.merge(df_perf[["EmployeeID","EnvironmentSatisfaction","JobSatisfaction",
                             "RelationshipSatisfaction","WorkLifeBalance","SelfRating",
                             "ManagerRating","TrainingOpportunitiesWithinYear",
                             "TrainingOpportunitiesTaken"]], on="EmployeeID")
df = df.merge(df_education, on="EducationLevelID", how="left")
print(f"Merged shape: {df.shape} | Missing: {df.isnull().sum().sum()}")
 

Merged shape: (2000, 38) | Missing: 0


In [11]:
# ## 3. Exploratory Data Analysis (EDA)
 
# %%
palette = {"Yes":"#ff4b4b","No":"#00c9b1"}
 
def dark_ax(ax):
    ax.set_facecolor('#1a1d2e'); ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#444466')
 
fig = plt.figure(figsize=(24,32))
fig.patch.set_facecolor('#0f1117')
 
ax1=fig.add_subplot(5,4,1)
la=df.groupby("Level")["Attrition"].apply(lambda x:(x=="Yes").mean()).reindex(levels)
ax1.barh(la.index,la.values,color="#7c6af7"); ax1.set_title("Attrition by Level"); dark_ax(ax1)
 
ax2=fig.add_subplot(5,4,2)
ga=df.groupby("Gender")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax2.bar(ga.index,ga.values,color=["#ff6b9d","#4b91ff","#a78bfa"])
ax2.set_title("By Gender"); dark_ax(ax2)
 
ax3=fig.add_subplot(5,4,3)
da=df.groupby("Department")["Attrition"].apply(lambda x:(x=="Yes").mean()).sort_values()
ax3.barh(da.index,da.values,color="#f59e0b"); ax3.set_title("By Department"); dark_ax(ax3)
 
ax4=fig.add_subplot(5,4,4)
la2=df.groupby("Location")["Attrition"].apply(lambda x:(x=="Yes").mean()).sort_values()
ax4.barh(la2.index,la2.values,color="#10b981"); ax4.set_title("By Location"); dark_ax(ax4)
 
ax5=fig.add_subplot(5,4,5)
for lbl,grp in df.groupby("Attrition"):
    ax5.hist(grp["MarketRateRatio"],bins=30,alpha=0.6,label=lbl,color=palette[lbl])
ax5.axvline(1.0,color="white",ls="--",lw=1); ax5.set_title("Market Rate Ratio")
ax5.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax(ax5)
 
ax6=fig.add_subplot(5,4,6)
for lbl,grp in df.groupby("Attrition"):
    ax6.hist(grp["YearsSinceLastPromotion"],bins=20,alpha=0.6,label=lbl,color=palette[lbl])
ax6.set_title("Yrs Since Promotion"); ax6.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax(ax6)
 
ax7=fig.add_subplot(5,4,7)
ot=df.groupby(["OverTime","Attrition"]).size().unstack(fill_value=0)
ot.div(ot.sum(axis=1),axis=0).plot(kind="bar",ax=ax7,color=[palette["No"],palette["Yes"]],rot=0)
ax7.set_title("OverTime vs Attrition"); ax7.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax(ax7)
 
ax8=fig.add_subplot(5,4,8)
sa=df.groupby("JobSatisfaction")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax8.bar(sa.index,sa.values,color="#818cf8"); ax8.set_title("Job Satisfaction"); dark_ax(ax8)
 
ax9=fig.add_subplot(5,4,9)
bt=df.groupby(["BusinessTravel","Attrition"]).size().unstack(fill_value=0)
bt.div(bt.sum(axis=1),axis=0)[["Yes"]].plot(kind="bar",ax=ax9,color="#f97316",rot=15)
ax9.set_title("Business Travel"); dark_ax(ax9)
 
ax10=fig.add_subplot(5,4,10)
aw=df.groupby("AwardsReceived")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax10.plot(aw.index,aw.values,"o-",color="#ec4899",lw=2)
ax10.set_title("Awards vs Attrition"); dark_ax(ax10)
 
ax11=fig.add_subplot(5,4,11)
for lbl,grp in df.groupby("Attrition"):
    ax11.hist(grp["Leave_Last90Days"],bins=15,alpha=0.6,label=lbl,color=palette[lbl])
ax11.set_title("Leaves (90d)"); ax11.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax(ax11)
 
ax12=fig.add_subplot(5,4,12)
mr=df.groupby("ManagerRating")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax12.bar(mr.index,mr.values,color="#06b6d4"); ax12.set_title("Manager Rating"); dark_ax(ax12)
 
ax13=fig.add_subplot(5,4,13)
for lbl,grp in df.groupby("Attrition"):
    ax13.hist(grp["Age"],bins=20,alpha=0.6,label=lbl,color=palette[lbl])
ax13.set_title("Age Distribution"); ax13.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax(ax13)
 
ax14=fig.add_subplot(5,4,14)
so=df.groupby("StockOptionLevel")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax14.bar(so.index,so.values,color="#84cc16"); ax14.set_title("Stock Option Level"); dark_ax(ax14)
 
ax15=fig.add_subplot(5,4,15)
wl=df.groupby("WorkLifeBalance")["Attrition"].apply(lambda x:(x=="Yes").mean())
ax15.bar(wl.index,wl.values,color="#fb923c"); ax15.set_title("Work-Life Balance"); dark_ax(ax15)
 
ax16=fig.add_subplot(5,4,(16,20))
nc=["Age","Salary","MarketRateRatio","YearsAtCompany","YearsSinceLastPromotion",
    "Leave_Last90Days","AwardsReceived","JobSatisfaction","WorkLifeBalance","ManagerRating"]
sns.heatmap(df[nc].corr(),ax=ax16,cmap="coolwarm",annot=True,fmt=".2f",
            annot_kws={"size":6},linewidths=0.3,cbar_kws={"shrink":0.6})
ax16.set_title("Feature Correlation Heatmap",color="white")
ax16.tick_params(colors='white',labelsize=6); ax16.set_facecolor('#1a1d2e')
 
fig.suptitle("TechNova Inc. – EDA Dashboard",color="white",fontsize=16,fontweight="bold",y=1.005)
plt.tight_layout()
plt.savefig("01_eda_dashboard.png",dpi=150,bbox_inches="tight",facecolor='#0f1117')
plt.show()
print("EDA plot saved ✓")
 

EDA plot saved ✓


In [12]:
# ## 4. Feature Engineering
 
# %%
df["TrainingUtilizationRate"] = df["TrainingOpportunitiesTaken"]/df["TrainingOpportunitiesWithinYear"].clip(lower=1)
df["AvgSatisfaction"]         = df[["EnvironmentSatisfaction","JobSatisfaction","RelationshipSatisfaction","WorkLifeBalance"]].mean(axis=1)
df["RatingGap"]               = df["ManagerRating"] - df["SelfRating"]
df["PromotionStagnation"]     = df["YearsSinceLastPromotion"]/(df["YearsAtCompany"]+0.1)
df["LeaveIntensity"]          = df["Leave_Last90Days"]/90
df["PayDeficit"]              = (1-df["MarketRateRatio"]).clip(lower=0)
df["DistanceBucket"]          = pd.cut(df["DistanceFromHome_KM"],bins=[0,10,25,50,101],
                                        labels=["Near","Medium","Far","Very Far"])
df["TenureBucket"]            = pd.cut(df["YearsAtCompany"],bins=[0,1,3,7,15,50],
                                        labels=["<1yr","1-3yr","3-7yr","7-15yr","15+yr"])
df["IsNewbie"]                = (df["YearsAtCompany"]<1).astype(int)
df["IsVeteran"]               = (df["YearsAtCompany"]>10).astype(int)
df["OverTimeFlag"]            = (df["OverTime"]=="Yes").astype(int)
print(f"Features after engineering: {df.shape[1]}")
 

Features after engineering: 49


In [14]:
# %%
from sklearn.preprocessing import LabelEncoder
target = "Attrition"
drop_cols = ["EmployeeID","FirstName","LastName","HireDate","Attrition","EducationLevel"]
feature_cols = [c for c in df.columns if c not in drop_cols]
 
X = df[feature_cols].copy()
y = (df[target]=="Yes").astype(int)
 
for c in X.select_dtypes(include=["object","category"]).columns:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))
 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (1600, 43) | Test: (400, 43)


In [20]:
# %%
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    average_precision_score
)
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
 
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000,class_weight="balanced",C=0.5),
    "Decision Tree":       DecisionTreeClassifier(max_depth=6,class_weight="balanced",random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced",random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200,learning_rate=0.05,max_depth=4,random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=9),
    "SVM (RBF)":           SVC(kernel="rbf",probability=True,class_weight="balanced",C=1.0),
}
 
results = {}
for name,model in models.items():
    scaled = name in ["Logistic Regression","K-Nearest Neighbors","SVM (RBF)"]
    Xtr,Xte = (X_train_s,X_test_s) if scaled else (X_train.values,X_test.values)
    cv_auc = cross_val_score(model,Xtr,y_train,cv=cv,scoring="roc_auc")
    model.fit(Xtr,y_train)
    yp,ypr = model.predict(Xte), model.predict_proba(Xte)[:,1]
    results[name] = {"model":model,"scaled":scaled,"y_pred":yp,"y_prob":ypr,
                      "cv_auc":cv_auc.mean(),"cv_auc_std":cv_auc.std(),
                      "test_auc":roc_auc_score(y_test,ypr),"test_f1":f1_score(y_test,yp),
                      "test_acc":accuracy_score(y_test,yp),"avg_prec":average_precision_score(y_test,ypr)}
    print(f"{name:25s} | CV AUC={cv_auc.mean():.3f}±{cv_auc.std():.3f} | Test AUC={results[name]['test_auc']:.3f} | F1={results[name]['test_f1']:.3f}")
 
best_name = max(results,key=lambda k:results[k]["test_auc"])
best = results[best_name]
print(f"\n🏆 Best: {best_name} | AUC={best['test_auc']:.3f}")

Logistic Regression       | CV AUC=0.608±0.033 | Test AUC=0.651 | F1=0.379
Decision Tree             | CV AUC=0.562±0.043 | Test AUC=0.562 | F1=0.298
Random Forest             | CV AUC=0.590±0.043 | Test AUC=0.619 | F1=0.000
Gradient Boosting         | CV AUC=0.567±0.041 | Test AUC=0.626 | F1=0.159
K-Nearest Neighbors       | CV AUC=0.559±0.036 | Test AUC=0.597 | F1=0.068
SVM (RBF)                 | CV AUC=0.595±0.031 | Test AUC=0.631 | F1=0.333

🏆 Best: Logistic Regression | AUC=0.651


In [21]:
# ## 7. Visualizations – Model Comparison
 
# %%
colors = ["#7c6af7","#f59e0b","#10b981","#f97316","#ec4899","#06b6d4"]
names  = list(results.keys())
 
def dark_ax2(ax):
    ax.set_facecolor('#1a1d2e'); ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#444466')
 
fig2,axes = plt.subplots(3,3,figsize=(22,18))
fig2.patch.set_facecolor('#0f1117')
 
# A: CV AUC
ax=axes[0,0]
ax.barh(names,[results[n]["cv_auc"] for n in names],
        xerr=[results[n]["cv_auc_std"] for n in names],
        color=colors,alpha=0.85,error_kw=dict(ecolor="white",capsize=3))
ax.axvline(0.7,color="white",ls="--",lw=0.8,alpha=0.5)
ax.set_title("Cross-Validated AUC (5-Fold)"); dark_ax2(ax)
 
# B: Test metrics
ax=axes[0,1]
x=np.arange(len(names)); w=0.2
for i,(m,ml) in enumerate(zip(["test_auc","test_f1","test_acc","avg_prec"],["AUC","F1","Acc","AP"])):
    ax.bar(x+i*w,[results[n][m] for n in names],w,label=ml,alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels([n.split()[0] for n in names],rotation=30,ha="right",color="white",fontsize=7)
ax.set_title("Test Metrics"); ax.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax2(ax)
 
# C: ROC Curves
ax=axes[0,2]
for (n,res),c in zip(results.items(),colors):
    fpr,tpr,_=roc_curve(y_test,res["y_prob"])
    ax.plot(fpr,tpr,color=c,lw=1.5,label=f"{n.split()[0]} ({res['test_auc']:.3f})")
ax.plot([0,1],[0,1],"w--",lw=0.8,alpha=0.4); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curves"); ax.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=6.5); dark_ax2(ax)
 
# D: PR Curve (best)
ax=axes[1,0]
prec,rec,_=precision_recall_curve(y_test,best["y_prob"])
ax.plot(rec,prec,color="#7c6af7",lw=2); ax.fill_between(rec,prec,alpha=0.15,color="#7c6af7")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"Precision-Recall\n({best_name})"); dark_ax2(ax)
 
# E: Confusion Matrix
ax=axes[1,1]
cm=confusion_matrix(y_test,best["y_pred"])
ax.imshow(cm,cmap="magma")
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=14,fontweight="bold",
                color="white" if cm[i,j]<cm.max()*0.7 else "black")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Stay","Leave"],color="white"); ax.set_yticklabels(["Stay","Leave"],color="white")
ax.set_title(f"Confusion Matrix\n({best_name})"); dark_ax2(ax)
 
# F: Feature Importance
ax=axes[1,2]
bm=best["model"]; fn=list(X.columns)
if hasattr(bm,"feature_importances_"):
    imp=bm.feature_importances_; si=np.argsort(imp)[-20:]
    ax.barh([fn[i] for i in si],imp[si],color="#7c6af7",alpha=0.85)
    ax.set_title(f"Feature Importance\n({best_name})")
elif hasattr(bm,"coef_"):
    coef=np.abs(bm.coef_[0]); si=np.argsort(coef)[-20:]
    ax.barh([fn[i] for i in si],coef[si],color="#f59e0b",alpha=0.85)
    ax.set_title(f"Coefficient Magnitude\n({best_name})")
ax.tick_params(colors='white',labelsize=6.5); dark_ax2(ax)
 
# G: Risk Score Distribution
ax=axes[2,0]
ax.hist(best["y_prob"][y_test==0],bins=30,alpha=0.6,color="#00c9b1",label="Stayed")
ax.hist(best["y_prob"][y_test==1],bins=30,alpha=0.6,color="#ff4b4b",label="Left")
ax.axvline(0.5,color="white",ls="--",lw=1); ax.set_xlabel("Predicted Probability")
ax.set_title("Risk Score Distribution"); ax.legend(facecolor="#1a1d2e",labelcolor="white",fontsize=7); dark_ax2(ax)
 
# H: Risk Bucketing
ax=axes[2,1]
rdf=pd.DataFrame({"prob":best["y_prob"],"actual":y_test.values})
rdf["bucket"]=pd.cut(rdf["prob"],bins=[0,.2,.4,.6,.8,1.0],
                      labels=["Very Low","Low","Medium","High","Very High"])
bkt=rdf.groupby("bucket",observed=True).agg(count=("prob","count"),rate=("actual","mean")).reset_index()
ax.bar(bkt["bucket"],bkt["count"],color=["#22c55e","#84cc16","#eab308","#f97316","#ef4444"],alpha=0.85)
ax2b=ax.twinx(); ax2b.plot(bkt["bucket"],bkt["rate"],"o--",color="white",lw=1.5,markersize=5)
ax2b.set_ylabel("Attrition Rate",color="white"); ax2b.tick_params(colors="white")
ax.set_title("Risk Bucket Distribution"); ax.set_ylabel("Count"); dark_ax2(ax)
 
# I: CV Variance Boxplot
ax=axes[2,2]
for i,(n,res) in enumerate(results.items()):
    Xtr=X_train_s if res["scaled"] else X_train.values
    sc=cross_val_score(res["model"],Xtr,y_train,cv=cv,scoring="roc_auc")
    ax.boxplot(sc,positions=[i],patch_artist=True,
               boxprops=dict(facecolor=colors[i],alpha=0.7),
               medianprops=dict(color="white",linewidth=2),
               whiskerprops=dict(color="white"),capprops=dict(color="white"),
               flierprops=dict(color=colors[i],markeredgecolor=colors[i]))
ax.set_xticks(range(len(results))); ax.set_xticklabels([n.split()[0] for n in names],rotation=30,ha="right",color="white",fontsize=7)
ax.set_ylabel("AUC"); ax.set_title("CV Score Variance"); dark_ax2(ax)
 
fig2.suptitle(f"TechNova – Model Comparison | Best: {best_name} (AUC={best['test_auc']:.3f})",
              color="white",fontsize=14,fontweight="bold")
plt.tight_layout()
plt.savefig("02_model_comparison.png",dpi=150,bbox_inches="tight",facecolor='#0f1117')
plt.show()
print("Model comparison plot saved ✓")

Model comparison plot saved ✓


In [22]:
# ## 8. Detailed Analysis & Classification Report
 
# %%
print(f"\n{'='*60}\n  {best_name} – Classification Report\n{'='*60}")
print(classification_report(y_test,best["y_pred"],target_names=["Stayed","Left"]))
 
# Apply predictions to full dataset
Xall = scaler.transform(X) if best["scaled"] else X.values
df["AttritionProb"] = best["model"].predict_proba(Xall)[:,1]
df["RiskCategory"]  = pd.cut(df["AttritionProb"],bins=[0,.25,.5,.75,1.0],
                              labels=["Low","Medium","High","Critical"])
 
for dim in ["Level","Department","Location","Gender","MaritalStatus","OverTime"]:
    print(f"\n── By {dim} ──")
    print(df.groupby(dim)["AttritionProb"].mean().sort_values(ascending=False).apply(lambda x:f"{x:.1%}"))
 
print("\n── Risk Distribution ──")
print(df["RiskCategory"].value_counts().sort_index())
 
print("\n── Top 10 Highest-Risk Employees ──")
top10 = df.nlargest(10,"AttritionProb")[["EmployeeID","FirstName","Department","Level","Location","AttritionProb","RiskCategory"]]
print(top10.to_string(index=False))
 


  Logistic Regression – Classification Report
              precision    recall  f1-score   support

      Stayed       0.87      0.63      0.73       323
        Left       0.28      0.60      0.38        77

    accuracy                           0.62       400
   macro avg       0.57      0.61      0.55       400
weighted avg       0.75      0.62      0.66       400


── By Level ──
Level
L3    49.3%
L1    47.4%
L2    47.0%
L4    46.4%
L5    40.9%
L6    29.8%
Name: AttritionProb, dtype: object

── By Department ──
Department
Engineering    49.0%
HR             47.7%
Finance        47.7%
Legal          46.6%
Operations     46.5%
Sales          44.1%
Marketing      43.8%
Name: AttritionProb, dtype: object

── By Location ──
Location
Hyderabad    48.4%
Delhi        48.1%
Pune         46.9%
Remote       46.7%
Bengaluru    46.4%
Mumbai       46.2%
Chennai      45.6%
Name: AttritionProb, dtype: object

── By Gender ──
Gender
Non-Binary    53.3%
Male          48.3%
Female        44.6%
Nam

In [23]:
 ## 9. Save All Artifacts
 
# %%
with open("model_artifacts.pkl","wb") as f:
    pickle.dump({"model":best["model"],"scaler":scaler,"feature_cols":list(X.columns),
                 "use_scaled":best["scaled"],"model_name":best_name,
                 "test_auc":best["test_auc"],"test_f1":best["test_f1"]},f)
 
df.to_csv("technova_hr_data.csv",index=False)
df_emp.to_csv("dim_employee.csv",index=False)
df_perf.to_csv("fact_performance.csv",index=False)
df_satisfaction.to_csv("dim_satisfaction.csv",index=False)
df_rating.to_csv("dim_rating.csv",index=False)
df_education.to_csv("dim_education.csv",index=False)
 
summary = {n:{k:float(v) if isinstance(v,(float,np.floating)) else v
              for k,v in res.items() if k not in ["model","y_prob","y_pred","scaled"]}
           for n,res in results.items()}
with open("model_summary.json","w") as f: json.dump(summary,f,indent=2)
 
print("All artifacts saved ✓")
print("  model_artifacts.pkl | technova_hr_data.csv | model_summary.json")
print("  dim_employee.csv | dim_education.csv | fact_performance.csv")
 

All artifacts saved ✓
  model_artifacts.pkl | technova_hr_data.csv | model_summary.json
  dim_employee.csv | dim_education.csv | fact_performance.csv
